In [31]:
import torch
import rasterio
import numpy as np
import os
import sklearn.decomposition
import torchgeo.models
import tqdm

import sys
sys.path.append('..')
import model_to_tif
from torch.utils.data import DataLoader
from torchgeo.datasets import RasterDataset, Sentinel2, stack_samples
from torchgeo.samplers import GridGeoSampler

import models

NOTE: This is on RBG images!

In [20]:
def fit_rcf_pca_context(train_sites,
                        subimage_size=[3,11,11],
                        rcf_features=64,
                        rcf_kernel_size=3,
                        n_per_image = 100,
                        img_by_site_dir = '../data/int/sentinel_by_site',
                       ):
    
    rcf = torchgeo.models.RCF(features = rcf_features, 
                              in_channels=subimage_size[0], 
                              kernel_size=rcf_kernel_size)

    rs = np.random.RandomState(0)
    X_by_site_id_sampled = []
    sites_sampled = []
    for s,site_id in enumerate(train_sites):
        # print(site_id)
        site_id = site_id
        vis_tifs = [x for x in os.listdir(os.path.join(img_by_site_dir, site_id)) if 'TCI' in x]
        with rasterio.open(os.path.join(img_by_site_dir, site_id, vis_tifs[0])) as f:
            data = f.read()

            for i in range(n_per_image):
                x0 = rs.randint(0,data.shape[1]-subimage_size[1])
                y0 = rs.randint(0,data.shape[2]-subimage_size[2])
                x1 = x0 + subimage_size[1]
                y1 = y0 + subimage_size[2]

                X = rcf(torch.Tensor(data[:,x0:x1, y0:y1]/ 255.))
                X_by_site_id_sampled.append(X.numpy())
                sites_sampled.append(s)

    X_by_site_id_sampled = np.array(X_by_site_id_sampled)
    
    pca = sklearn.decomposition.PCA()
    X_sampled_pca = pca.fit_transform(X_by_site_id_sampled)
    
    return rcf, pca

def rcf_pca_context(X_in, rcf, pca):
    # returns first n dimensions of X depending on the PCA embedding of training RCFs
    
    # TODO: make sure that if X_in is preprocessed that we use the same preprocessing in the context model.
    X_pca = pca.transform(rcf(X_in / 255.))
    
    return X_pca

def my_transforms_3_channel_rgb_plus_mask(sample):
     
    if len(sample['image']) > 5:
        sample['vis'] = sample['image'][4:7]
        
    label_band = len(sample['image']) - 1 
    sample['mask'] = torch.Tensor(sample['image'][label_band]).clone()
    sample['image'] = sample['vis']
    return sample
    
    

# Now try to train a model this way...


In [21]:
from importlib import reload

import pixelwise_regression_task_with_mask
reload(pixelwise_regression_task_with_mask)
from pixelwise_regression_task_with_mask import RegressionTaskWithMask
from lightning.pytorch import Trainer
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint
from lightning.pytorch.loggers import TensorBoardLogger
from torch import Tensor
import torch.nn as nn

In [22]:
site_id = 'KaringaniSite15'
eval_input_dir = f'/n/home10/erolf/tree_mapping/data/int/sentinel_by_site/{site_id}'

context_size = 11
subimage_size = [3, 64, 64]

# 1. train PCA on all since its unsupervised
train_sites_within_S2_tile = [
    ['KaringaniVultureSite01'],
    ['KaringaniSite11DevNodeF'],
    ['KaringaniSouthSouthDevNode'],
    ['KaringaniSite15'],
    ['KaringaniSite06'],
    ['KaringaniSite12'],
    ['KaringaniSite16'],
    ['KaringaniSite13'],
]

all_sites = [x[0] for x in train_sites_within_S2_tile]

rcf, pca = fit_rcf_pca_context(all_sites, subimage_size=subimage_size )

device='cuda'
rcf = rcf.to(device)

site_id_train = 'KaringaniSite15'
site_id_val = 'KaringaniSite12'
site_id_test = 'KaringaniSite12'

train_sites = ['KaringaniSite15', 'KaringaniSite16','KaringaniSite11DevNodeF', 'KaringaniSite06']
val_sites = ['KaringaniSouthSouthDevNode', 'KaringaniSite12' , 'KaringaniSite14','KaringaniVultureSite01']
test_sites = ['KaringaniSouthSouthDevNode', 'KaringaniSite12', 'KaringaniSite14','KaringaniVultureSite01']

In [23]:
subimage_size

[3, 64, 64]

In [24]:
# import time
# %time rcf, pca = fit_rcf_pca_context(all_sites, subimage_size=subimage_size, n_per_image = 100)
# rcf = rcf.to(device)

In [14]:
reload(pixelwise_regression_task_with_mask)

            



In [25]:
batch_size = 32
patch_size = 64
length= 10000
 
import chm_datamodule
reload(chm_datamodule)
from chm_datamodule import ChmDataModule, my_transforms_4_channel_rgbnir_plus_mask, image_stats

chm = chm_datamodule.ChmDataModule(train_sites=train_sites, 
                    val_sites=val_sites,
                    test_sites=test_sites, 
                                batch_size=batch_size, 
                    patch_size=patch_size,
                                length=length,
                                batch_transforms = my_transforms_3_channel_rgb_plus_mask,
                    num_workers=1,
                               )

RCF()

In [66]:
lr = 0.001
max_epochs= 50
num_layers_fcn = 5

conditions = [
   {'context': True, 'num_filters': 16, 'context_type': 'fov'}, 
   {'context': True, 'num_filters': 16, 'context_type': 'patch'}, 
]



In [67]:
from importlib import reload
reload(models)

<module 'models' from '/n/home10/erolf/tree_mapping/experiment_notebooks/../models.py'>

In [ ]:
exp_name = 'experiment_3'
    
for path in [exp_name, f"{exp_name}/logs", f"{exp_name}/models"]:
    if not os.path.exists(path): os.mkdir(path)
        
for condition in conditions:
    num_filters = condition['num_filters']
    use_context = condition['context']
    context_type = condition['context_type']
    
    task = models.ContextualizedPixelwiseRegressionTask(loss='mse',
                                                        learning_rate=lr,
                                                        learning_rate_schedule_patience=10,
                                                             use_context=use_context,
                                                        weights=None,
                                                        in_channels=3,
                                                        num_classes=1, 
                                                        num_layers_fcn=num_layers_fcn,
                                                        context_type=context_type,
                                                        num_filters=num_filters,)

    


    # Set up callbacks
    accelerator = "gpu" if torch.cuda.is_available() else "cpu"
    default_root_dir = os.path.join(exp_name)
    checkpoint_callback = ModelCheckpoint(
                    monitor="val_loss", dirpath=default_root_dir, save_top_k=1, save_last=True
    )

    # save name
    version_id = f"contexttype_{context_type}_filters_{num_filters}_lr_{lr}_train_sites" 
    for x in train_sites:
        version_id += f"_{x}" 

    logger = TensorBoardLogger(
        save_dir=default_root_dir, name="logs", version=version_id)
    print(f"logs will go in {default_root_dir}/logs")


    trainer = Trainer(
                    accelerator=accelerator,
                    callbacks=[checkpoint_callback, 
                    #          early_stopping_callback
                             ],
                    fast_dev_run=False,
                    log_every_n_steps=1,
                    logger=logger,
                    min_epochs=1,
                    max_epochs=max_epochs,
    )

    trainer.fit(model=task, datamodule=chm)
    
    model_save_path = f'{exp_name}/models/{version_id}.net'
    torch.save(task.model.state_dict(), model_save_path)
        

logs will go in experiment_3/logs


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs


['r', 'g', 'b', 'nir', 'vis']
Converting RasterDataset resolution from 9.995824634655532 to 10
['r', 'g', 'b', 'nir', 'vis']
Converting RasterDataset resolution from 9.98956158663883 to 10
['r', 'g', 'b', 'nir', 'vis']
Converting RasterDataset resolution from 10.003759398496241 to 10
['r', 'g', 'b', 'nir', 'vis']
Converting RasterDataset resolution from 9.995824634655532 to 10
['r', 'g', 'b', 'nir', 'vis']
Converting RasterDataset resolution from 9.992831541218639 to 10
['r', 'g', 'b', 'nir', 'vis']
['r', 'g', 'b', 'nir', 'vis']


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name          | Type                       | Params
-------------------------------------------------------------
0 | model         | ContextModelPrototypeOuter | 19.6 K
1 | loss          | MSELoss                    | 0     
2 | train_metrics | MetricCollection           | 0     
3 | val_metrics   | MetricCollection           | 0     
4 | test_metrics  | MetricCollection           | 0     
-------------------------------------------------------------
19.6 K    Trainable params
0         Non-trainable params
19.6 K    Total params
0.079     Total estimated model params size (MB)


Converting RasterDataset resolution from 10.006263048016702 to 10
['r', 'g', 'b', 'nir', 'vis']
Converting RasterDataset resolution from 10.006944444444445 to 10


...

Sanity Checking: 0it [00:00, ?it/s]

Training: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

In [ ]:
2+2

2